# 03 · Segment-wise (leaky) vs subject-wise splits

**Goal:** many published studies cut each recording into short windows (here 4 s), shuffle *all* windows together, and then split them into training and test sets. Windows from the same participant then end up on both sides. Here we copy that approach (a **segment-wise split (leaky)**) and compare it with a **subject-wise split**. Both use the same features and the same model.

**How it fits the project:** this is the core question. Do the high published accuracies survive a subject-wise split? Any gap between the two scores is **leakage**: the model has partly seen the test participants during training.

Analogy: an exam whose questions are slightly reworded homework questions. A high score shows memory of the homework, not understanding.

**Question:** How many 4-second windows do we get, and how many per group?

**How:**
1. For each participant, `window_features` (in `src/features.py`):
   - band-pass filters the whole recording at 1–40 Hz,
   - cuts it into non-overlapping 4-second segments (512 samples) and drops the leftover at the end,
   - computes the same 76 features as notebook 02 for each segment. With `n_fft=512` the whole 4 s segment is a single Welch window, so each spectrum is noisier than the per-participant one.
2. Add the participant's `subject_id` and `label` to every window row. `pd.concat` stacks all participants into one big table.
3. `X` = features, `y` = labels, `g` = **groups** (subject IDs). Groups tell scikit-learn which windows belong to the same participant.

**What we found:** 4173 windows × 78 columns (76 features + `subject_id` + `label`). **Control 1843, ADHD 2330.** ADHD has more windows because its recordings are longer (notebook 01). Always guessing "ADHD" would already score 2330 ÷ 4173 = **0.56 accuracy**, the baseline for accuracy here.

In [ ]:
import sys; sys.path.append("..")
import pandas as pd
from src.load import list_subjects, load_subject
from src.features import window_features

parts = []
for _, k in pd.DataFrame(list_subjects()).iterrows():
    f = window_features(load_subject(k.path))
    f["subject_id"], f["label"] = k.subject_id, k.label
    parts.append(f)
win = pd.concat(parts, ignore_index=True)

X, y, g = win.drop(columns=["subject_id", "label"]), win.label, win.subject_id
print(win.shape)
print(win.groupby("label").size())

**Question:** Does the way we split the data change the score?

**How:**
1. **Random forest**: 300 decision trees. Each tree is trained on a random sample of rows and features, and they vote. `n_jobs=-1` uses all CPU cores.
2. Two ways to split into 5 folds:
   - `StratifiedKFold(shuffle=True)`, the **segment-wise split (leaky)**: windows are shuffled at random, keeping the ADHD/Control mix. It ignores which participant a window came from, so each participant's windows land in both training and test data. The warning *"The groups parameter is ignored by StratifiedKFold"* confirms that it does not look at participants.
   - `StratifiedGroupKFold(shuffle=True)`, the **subject-wise split**: all windows of a participant stay together in one fold, while it tries to keep the class mix balanced. A participant is either fully in training or fully in test.
3. `cross_validate` returns accuracy and AUC for each fold. We print the means.

**What we found:** | split | accuracy | AUC |
|---|---|---|
| segment-wise split (leaky) | **0.79** | **0.88** |
| subject-wise split | **0.63** | **0.64** |

Same data, same features, same model; only the split differs. The drop (0.16 accuracy, 0.24 AUC) is **leakage**. In the segment-wise split, the model trains on other windows of the very same participant and can recognise them (see the last cell). Subject-wise accuracy (0.63) is only a little above the 0.56 always-ADHD baseline. Subject-wise AUC (0.64) is about the same as the recording-duration baseline (0.65, notebook 01).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_validate

model = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
splits = {
    "segment-wise split (leaky)": StratifiedKFold(5, shuffle=True, random_state=42),
    "subject-wise split":         StratifiedGroupKFold(5, shuffle=True, random_state=42),
}
for name, cv in splits.items():
    r = cross_validate(model, X, y, groups=g, cv=cv, scoring=["accuracy", "roc_auc"])
    print(f"{name:28s} accuracy {r['test_accuracy'].mean():.2f}   AUC {r['test_roc_auc'].mean():.2f}")

**Question:** If we combine all windows of a participant into one score per participant, how good is the subject-wise model?

**How:**
1. `cross_val_predict` gives every window a prediction from the fold in which its participant was in the **test** set, so it stays subject-wise.
2. `method="predict_proba"` returns probabilities. `[:, 1]` is the probability of ADHD.
3. Group the windows by participant and **average** their probabilities, giving one score per participant (**subject-level aggregation**). This is like averaging several measurements from the same patient.
4. `roc_auc_score` computes AUC over the 121 participants.

**What we found:** **Per-subject AUC 0.70.** This is higher than the window-level 0.64, because averaging smooths out noisy single windows. It is still subject-wise. However, it comes from **one** 5-fold split (one `random_state`) of only 121 participants, and a different split could move it by several points. So we cannot yet say it really beats the recording-duration baseline (0.65). Repeating it with several random splits would tell.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

subwise = StratifiedGroupKFold(5, shuffle=True, random_state=42)
p = cross_val_predict(model, X, y, groups=g, cv=subwise, method="predict_proba")[:, 1]
per_subject = pd.DataFrame({"subject_id": g, "label": y, "p": p}).groupby("subject_id").mean()
print("Subject-wise, per-subject AUC:", round(roc_auc_score(per_subject.label, per_subject.p), 2))

**Question:** Why does the segment-wise split score so high? Can the model tell *which participant* a window came from?

**How:**
1. Same random forest, but the target is now `g` (the subject ID) instead of ADHD/Control, so there are 121 possible answers.
2. Use the segment-wise shuffled split, so every participant has windows in both training and test data.
3. Accuracy = fraction of test windows assigned to the correct participant. Random guessing would give 1 ÷ 121 ≈ 0.008.

**What we found:** **0.70**, about 85 times better than guessing. Each participant's EEG carries a strong personal **"fingerprint"**, like a voice you recognise on the phone. Every participant has one fixed label, so recognising the participant is enough to know the label. This is how leakage inflates the segment-wise score: the model can recognise *who* it is rather than learn what ADHD looks like.

In [ ]:
from sklearn.model_selection import cross_val_score
probe = cross_val_score(model, X, g, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="accuracy")
print(f"Subject-identification probe accuracy: {probe.mean():.2f}  (guessing = 0.008)")

## Key takeaways
- Same data, features and model: segment-wise split (leaky) **0.79 accuracy / 0.88 AUC**; subject-wise split **0.63 / 0.64**.
- The gap is leakage. With shuffled windows, the same participant appears in both training and test data.
- A model can identify the participant from a single 4-second window 70% of the time (chance 0.8%). This strong fingerprint explains the leakage.
- The subject-wise window-level AUC (0.64) is close to the recording-duration baseline (0.65). Averaging per participant gives 0.70, but from a single split.
- **Answer so far:** on this dataset, the high accuracies from shuffled windows do **not** survive a subject-wise split.

## Open questions
- Does the per-subject AUC of 0.70 hold over repeated random splits, and does it reliably beat the recording-duration baseline (0.65)?
- How much do recording length and window count (more windows for ADHD) affect the subject-wise result? Try equal length or an equal number of windows per participant.
- Is the participant "fingerprint" driven by artifacts (EMG at T8, EOG) or by genuine brain rhythms?
- Would a deep-learning model on raw windows, as many papers use, show an even bigger gap?